## 0. Импорты
Все библиотеки подключаем в одном месте вверху, так проще понять, что нужно установить.

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score
from catboost import CatBoostClassifier

# Фиксируем случайность, чтобы при каждом запуске получались одни и те же цифры
RANDOM_STATE = 42

In [34]:
df = pd.read_csv("data\WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Строк и столбцов:", df.shape)
df.head()

Строк и столбцов: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [36]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Пропусков в TotalCharges:", df["TotalCharges"].isna().sum())

# Смотрим, что это за клиенты
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges"]]

Пропусков в TotalCharges: 11


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


In [37]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])

In [38]:
df["Churn"].value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby("Contract")["Churn"].mean().sort_values().plot(kind="bar", ax=axes[0])
axes[0].set_title("Доля ушедших по типу контракта")
axes[0].set_ylabel("доля ушедших")

sns.histplot(data=df, x="tenure", hue="Churn", bins=30, ax=axes[1])
axes[1].set_title("Сколько месяцев клиент с нами (1 = ушёл)")

plt.tight_layout()
plt.show()

In [40]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Текстовые колонки (пол, тип контракта и т.д.) — категориальные признаки
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
print("Категориальные признаки:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("train:", X_train.shape, " test:", X_test.shape)

Категориальные признаки: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
train: (5634, 19)  test: (1409, 19)


Чтобы не повторять один и тот же код для каждой модели, пишем функцию оценки:
- **ROC-AUC** — насколько хорошо модель отделяет уходящих от остающихся (0.5 = монетка, 1.0 = идеально). Главная метрика здесь.
- **recall** — какую долю реально ушедших модель нашла. Для бизнеса это важно: пропущенный клиент = потерянные деньги.
- **accuracy** — доля верных ответов. Оставляем только чтобы показать, как она обманывает.

In [41]:
results = []  # сюда складываем результаты всех моделей


def evaluate(name, model, X_te):
    proba = model.predict_proba(X_te)[:, 1]  # вероятность, что клиент уйдёт
    pred = (proba >= 0.5).astype(int)        # порог 0.5: выше — «уйдёт»
    row = {
        "модель": name,
        "ROC-AUC": roc_auc_score(y_test, proba),
        "recall": recall_score(y_test, pred),
        "accuracy": accuracy_score(y_test, pred),
    }
    results.append(row)
    return row

In [42]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
evaluate("Baseline: никто не уйдёт", dummy, X_test)

{'модель': 'Baseline: никто не уйдёт',
 'ROC-AUC': 0.5,
 'recall': 0.0,
 'accuracy': 0.7345635202271115}

In [43]:
X_train_num = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_num = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
# На всякий случай выравниваем колонки test под train
X_test_num = X_test_num.reindex(columns=X_train_num.columns, fill_value=0)

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train_num, y_train)
evaluate("Логистическая регрессия", logreg, X_test_num)

{'модель': 'Логистическая регрессия',
 'ROC-AUC': 0.841778397788628,
 'recall': 0.5668449197860963,
 'accuracy': 0.8069552874378992}

In [44]:
cb_model = CatBoostClassifier(
    iterations=500,       # сколько деревьев построить
    learning_rate=0.05,   # насколько сильно каждое новое дерево исправляет ошибки
    depth=4,              # глубина деревьев
    auto_class_weights="Balanced",
    cat_features=cat_cols,
    random_state=RANDOM_STATE,
    verbose=0,            # не печатать лог обучения
)
cb_model.fit(X_train, y_train)
evaluate("CatBoost", cb_model, X_test)

{'модель': 'CatBoost',
 'ROC-AUC': 0.8451807073290449,
 'recall': 0.7780748663101604,
 'accuracy': 0.7572746628814763}

In [45]:
pd.DataFrame(results).round(3)

,модель,ROC-AUC,recall,accuracy
0,Baseline: никто не уйдёт,0.500,0.000,0.735
1,Логистическая регрессия,0.842,0.567,0.807
2,CatBoost,0.845,0.778,0.757


In [46]:
X_num = pd.get_dummies(X, columns=cat_cols, drop_first=True)
scores = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    X_num, y, cv=5, scoring="roc_auc",
)
print("ROC-AUC по фолдам:", scores.round(3))
print(f"Среднее: {scores.mean():.3f} ± {scores.std():.3f}")

ROC-AUC по фолдам: [0.858 0.859 0.833 0.838 0.839]
Среднее: 0.845 ± 0.011


In [ ]:
importance = pd.Series(cb_model.get_feature_importance(), index=X_train.columns).sort_values()
importance.plot(kind="barh", figsize=(8, 6), title="Важность признаков (CatBoost)")
plt.show()

In [48]:
proba_test = cb_model.predict_proba(X_test)[:, 1]
missed = X_test[(y_test == 1) & (proba_test < 0.5)]
print("Ушедших в test:", (y_test == 1).sum(), " из них модель пропустила:", len(missed))
missed.head(10)

Ушедших в test: 374  из них модель пропустила: 83


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
2488,Male,0,No,No,31,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,Yes,Electronic check,55.25,1715.65
523,Female,0,No,No,23,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,No,Bank transfer (automatic),75.60,1758.60
5086,Male,0,No,No,45,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Electronic check,20.40,930.45
7011,Female,0,No,No,4,Yes,No,DSL,Yes,Yes,No,Yes,No,No,Month-to-month,Yes,Mailed check,60.40,272.15
3158,Male,0,Yes,No,18,Yes,No,DSL,No,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,49.55,878.35
5968,Female,0,No,No,12,Yes,Yes,DSL,No,No,No,Yes,Yes,Yes,One year,No,Mailed check,74.05,872.65
1577,Female,0,Yes,Yes,17,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,Yes,One year,No,Bank transfer (automatic),98.60,1704.95
1014,Male,0,Yes,Yes,18,Yes,No,DSL,No,No,Yes,No,No,Yes,Month-to-month,No,Electronic check,57.45,990.85
3721,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.65,38.70
3240,Female,0,Yes,No,54,Yes,Yes,Fiber optic,Yes,Yes,No,No,Yes,Yes,One year,No,Bank transfer (automatic),106.55,5763.30
